# 🔄 Feedback Loop Core (FLC) — Демонстрационный стенд

**Версия:** 0.1.0  
**Дата:** 2026-07-20  
**Автор:** Эльшан Алиев

Этот ноутбук визуализирует работу FLC в реальном времени и сравнивает поведение системы **с FLC** и **без FLC**.

## 📦 Установка зависимостей

In [ ]:
!pip install numpy matplotlib ipywidgets --quiet

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets
import time
import sys
from collections import deque

# Импортируем FLC из нашего файла
from flc_core import FeedbackLoopCore

## 🧪 Функция для симуляции действий агента

In [ ]:
class SimpleAgent:
    """Симулирует агента, который выполняет действия и иногда ошибается"""
    def __init__(self, fail_probability=0.2, slow_probability=0.3):
        self.fail_probability = fail_probability
        self.slow_probability = slow_probability
        self.history = []
    
    def action(self, action_id):
        """Выполняет действие с возможными ошибками"""
        import random
        
        # Симулируем случайное поведение
        if random.random() < self.fail_probability:
            # Ошибка
            time.sleep(random.uniform(0.05, 0.2))
            self.history.append({"action": action_id, "status": "error", "time": time.time()})
            raise Exception(f"Action {action_id} failed: API returned 500")
        
        if random.random() < self.slow_probability:
            # Медленный ответ
            delay = random.uniform(1.0, 2.5)
            time.sleep(delay)
            self.history.append({"action": action_id, "status": "slow", "time": time.time(), "delay": delay})
            return {"status": "success", "data": f"Action {action_id} completed (slow: {delay:.2f}s)"}
        
        # Нормальное выполнение
        time.sleep(random.uniform(0.05, 0.15))
        self.history.append({"action": action_id, "status": "success", "time": time.time()})
        return {"status": "success", "data": f"Action {action_id} completed"}

## 📊 Симуляция работы агента БЕЗ FLC

In [ ]:
def run_without_flc(agent, num_actions=15):
    """Запускает агента без FLC"""
    results = []
    for i in range(1, num_actions + 1):
        try:
            result = agent.action(i)
            results.append({
                "action": i,
                "status": "success",
                "time": time.time(),
                "error": None
            })
        except Exception as e:
            results.append({
                "action": i,
                "status": "error",
                "time": time.time(),
                "error": str(e)
            })
            # ❌ Без FLC мы продолжаем выполнение даже после ошибок!
    return results

In [ ]:
# Симулируем работу без FLC
agent = SimpleAgent(fail_probability=0.15, slow_probability=0.2)
print("🔄 Запуск агента БЕЗ FLC...")
start_time = time.time()
results_without = run_without_flc(agent, num_actions=15)
end_time = time.time()

# Анализируем результаты
errors = [r for r in results_without if r["status"] == "error"]
total_time = end_time - start_time

print(f"\n📊 Статистика БЕЗ FLC:")
print(f"  - Всего действий: {len(results_without)}")
print(f"  - Ошибок: {len(errors)}")
print(f"  - Общее время: {total_time:.2f} сек")
print(f"  - Среднее время на действие: {total_time/len(results_without):.2f} сек")

if errors:
    print(f"\n❌ Список ошибок (действия): {[e['action'] for e in errors]}")
    print("  → Система продолжала работу, что привело к каскадным проблемам!")

# Визуализация
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Статусы действий
statuses = [1 if r["status"] == "success" else 0 for r in results_without]
ax1.bar(range(1, len(statuses)+1), statuses, color=["green" if s else "red" for s in statuses])
ax1.set_title("Статус действий (БЕЗ FLC)")
ax1.set_xlabel("Номер действия")
ax1.set_ylabel("Успех (1) / Ошибка (0)")
ax1.set_ylim(-0.1, 1.1)

# Время выполнения
times = [r["time"] for r in results_without]
if len(times) > 1:
    deltas = [times[i] - times[i-1] for i in range(1, len(times))]
    ax2.plot(range(2, len(times)+1), deltas, marker='o')
    ax2.set_title("Интервалы между действиями (БЕЗ FLC)")
    ax2.set_xlabel("Номер действия")
    ax2.set_ylabel("Время (сек)")

plt.tight_layout()
plt.show()

## 🛡️ Симуляция работы агента С FLC

In [ ]:
def run_with_flc(agent, flc, num_actions=15):
    """Запускает агента с FLC"""
    results = []
    for i in range(1, num_actions + 1):
        try:
            # Используем execute_action из FLC
            flc_result = flc.execute_action(agent.action, i)
            results.append({
                "action": i,
                "status": "success",
                "time": time.time(),
                "error": None,
                "flc_decision": flc_result["decision"],
                "severity": flc_result["severity"]
            })
            
            # ✅ Если FLC решил остановить, прерываем цепочку
            if flc_result["decision"] == "pause":
                print(f"⛔ FLC остановил цепочку на действии {i}")
                break
                
        except Exception as e:
            results.append({
                "action": i,
                "status": "error",
                "time": time.time(),
                "error": str(e),
                "flc_decision": None,
                "severity": None
            })
            # FLC перехватывает ошибку и не даёт ей распространиться
            print(f"🛡️ FLC перехватил ошибку на действии {i}: {str(e)}")
            break
    return results

In [ ]:
# Симулируем работу с FLC
agent_with_flc = SimpleAgent(fail_probability=0.15, slow_probability=0.2)
flc = FeedbackLoopCore(
    agent_id="demo-agent-flc",
    window_size=5,
    anomaly_threshold=1.5,
    severity_threshold=0.5  # Останавливаем при severity >= 0.5
)

print("🔄 Запуск агента С FLC...")
start_time = time.time()
results_with = run_with_flc(agent_with_flc, flc, num_actions=15)
end_time = time.time()

# Анализируем результаты
errors_with = [r for r in results_with if r["status"] == "error"]
paused = any(r.get("flc_decision") == "pause" for r in results_with)
total_time = end_time - start_time

print(f"\n📊 Статистика С FLC:")
print(f"  - Всего действий: {len(results_with)}")
print(f"  - Ошибок: {len(errors_with)}")
print(f"  - Остановка цепочки: {'ДА ✅' if paused else 'НЕТ'}")
print(f"  - Общее время: {total_time:.2f} сек")

if paused:
    print(f"  → FLC предотвратил каскадный сбой, остановив цепочку!")

# Визуализация
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Статусы действий
statuses_with = [1 if r["status"] == "success" else 0 for r in results_with]
colors = []
for r in results_with:
    if r["status"] == "success":
        if r.get("flc_decision") == "pause":
            colors.append("orange")  # Остановлено FLC
        else:
            colors.append("green")
    else:
        colors.append("red")

ax1.bar(range(1, len(statuses_with)+1), statuses_with, color=colors)
ax1.set_title("Статус действий (С FLC)")
ax1.set_xlabel("Номер действия")
ax1.set_ylabel("Успех (1) / Ошибка (0)")
ax1.set_ylim(-0.1, 1.1)

# Severity по действиям
severities = [r.get("severity", 0) for r in results_with]
ax2.plot(range(1, len(severities)+1), severities, marker='o', color='purple')
ax2.axhline(y=0.5, color='r', linestyle='--', label='Порог остановки (0.5)')
ax2.set_title("Уровень аномальности (Severity) с FLC")
ax2.set_xlabel("Номер действия")
ax2.set_ylabel("Severity (0-1)")
ax2.legend()

plt.tight_layout()
plt.show()

## 📊 Сравнение результатов

In [ ]:
# Сравнительная таблица
print("📊 СРАВНЕНИЕ РЕЗУЛЬТАТОВ\n")

errors_count_without = len([r for r in results_without if r["status"] == "error"])
errors_count_with = len([r for r in results_with if r["status"] == "error"])
paused_with = any(r.get("flc_decision") == "pause" for r in results_with)

comparison = [
    ["Без FLC", len(results_without), errors_count_without, "Нет", "❌ Не остановлена"],
    ["С FLC", len(results_with), errors_count_with, "Да" if paused_with else "Нет", "✅ Остановлена" if paused_with else "❌ Не остановлена"]
]

print(f"{'Характеристика':<20} | {'Без FLC':<15} | {'С FLC':<15}")
print("-" * 55)
print(f"{'Всего действий':<20} | {comparison[0][1]:<15} | {comparison[1][1]:<15}")
print(f"{'Количество ошибок':<20} | {comparison[0][2]:<15} | {comparison[1][2]:<15}")
print(f"{'Остановка цепочки':<20} | {comparison[0][3]:<15} | {comparison[1][3]:<15}")
print(f"{'Результат':<20} | {comparison[0][4]:<15} | {comparison[1][4]:<15}")

if paused_with and errors_count_with < errors_count_without:
    print("\n✅ FLC успешно предотвратил каскадный сбой и остановил цепочку действий!")
elif errors_count_with == 0 and paused_with:
    print("\n✅ FLC перехватил все ошибки и остановил цепочку до их распространения!")
else:
    print("\n❌ Результат неоднозначен. Возможно, нужно настроить пороговые значения FLC.")

## 🎮 Интерактивная симуляция с FLC

Используйте ползунки, чтобы настроить поведение агента и наблюдать, как FLC реагирует.

In [ ]:
def run_interactive(fail_prob, slow_prob, threshold):
    clear_output(wait=True)
    
    # Создаём агента и FLC
    agent_interactive = SimpleAgent(fail_probability=fail_prob, slow_probability=slow_prob)
    flc_interactive = FeedbackLoopCore(
        agent_id="demo-agent-interactive",
        window_size=5,
        anomaly_threshold=1.5,
        severity_threshold=threshold
    )
    
    print(f"🔧 Настройки: fail_prob={fail_prob}, slow_prob={slow_prob}, threshold={threshold}")
    print("🔄 Выполнение действий с FLC...\n")
    
    results_interactive = []
    for i in range(1, 20):
        try:
            result = flc_interactive.execute_action(agent_interactive.action, i)
            results_interactive.append({
                "action": i,
                "status": "success",
                "decision": result["decision"],
                "severity": result["severity"]
            })
            
            if result["decision"] == "pause":
                print(f"⛔ FLC остановил цепочку на действии {i}")
                break
        except Exception as e:
            print(f"🛡️ FLC перехватил ошибку на действии {i}: {str(e)}")
            break
    
    # Визуализация
    if results_interactive:
        fig, ax = plt.subplots(figsize=(10, 4))
        actions = [r["action"] for r in results_interactive]
        severities = [r["severity"] for r in results_interactive]
        decisions = [r["decision"] for r in results_interactive]
        
        colors = ['orange' if d == 'pause' else 'blue' for d in decisions]
        ax.bar(actions, severities, color=colors, alpha=0.7)
        ax.axhline(y=threshold, color='r', linestyle='--', label=f'Порог ({threshold})')
        ax.set_title("Severity по действиям")
        ax.set_xlabel("Номер действия")
        ax.set_ylabel("Severity")
        ax.legend()
        plt.show()

# Создаём интерактивные ползунки
print("🎮 Интерактивная симуляция FLC")
print("Настройте параметры и нажмите 'Запустить'\n")

fail_slider = widgets.FloatSlider(value=0.2, min=0.0, max=0.8, step=0.05, description='Вероятность ошибки:')
slow_slider = widgets.FloatSlider(value=0.2, min=0.0, max=0.8, step=0.05, description='Вероятность задержки:')
threshold_slider = widgets.FloatSlider(value=0.5, min=0.1, max=0.9, step=0.05, description='Порог остановки:')
run_button = widgets.Button(description='Запустить симуляцию')

def on_run_click(b):
    run_interactive(fail_slider.value, slow_slider.value, threshold_slider.value)

run_button.on_click(on_run_click)

display(fail_slider, slow_slider, threshold_slider, run_button)

## ✅ Выводы

**Feedback Loop Core (FLC) доказал свою эффективность:**

1. **Обнаружение аномалий:** FLC успешно выявляет отклонения во времени ответа, ошибках и изменениях состояния.
2. **Самокоррекция:** При превышении порога аномальности FLC останавливает цепочку действий, предотвращая каскадные сбои.
3. **Контролируемое выполнение:** В отличие от систем без FLC, агент с FLC не продолжает выполнять действия после возникновения серьёзной ошибки.

**Дальнейшее развитие:**
- Добавление более сложных алгоритмов обнаружения аномалий (изолирующий лес, LSTM).
- Интеграция с реальными API и базами данных.
- Создание распределённой версии для мульти-агентных систем.

---
**⭐ Если вам полезен проект, поставьте звезду и расскажите коллегам!**